<a href="https://colab.research.google.com/github/vtk251zhdo/Introduction_to_neural_networks/blob/LAB_4/%D0%9B%D0%90%D0%91%D0%9E%D0%A0%D0%90%D0%A2%D0%9E%D0%A0%D0%9D%D0%90_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Імпорт бібліотек
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input
from sklearn.model_selection import train_test_split

In [15]:
# Підготовка датасету
sentences = [
    # Позитивні
    "Я дуже задоволений результатом", "Це чудовий сервіс", "Неймовірно гарна погода",
    "Мені дуже подобається цей фільм", "Найкращий день у моєму житті", "Це просто супер",
    # Негативні
    "Це найгірший досвід у моєму житті", "Жахливий сервіс та грубий персонал",
    "Мені зовсім не подобається ця їжа", "Це було дуже погано і нудно",
    "Я розчарований якістю товару", "Просто огидно",
    # Нейтральні
    "Сьогодні звичайний робочий день", "Книга стоїть на полиці", "Поїзд прибуває вчасно",
    "Я купив хліб у магазині", "На вулиці хмарно", "Він іде по дорозі"
]

labels = [2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]

In [16]:
# Токенізація
vocab_size = 500
max_length = 15
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

labels_final = tf.keras.utils.to_categorical(labels, num_classes=3)

In [17]:
# Розділення даних
X_train, X_test, y_train, y_test = train_test_split(padded, labels_final, test_size=0.2, random_state=42)

In [18]:
# Реалізація LSTM
model = Sequential([
    Input(shape=(max_length,)),
    Embedding(vocab_size, 32),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [19]:
# Навчання
print("Початок навчання...")
model.fit(X_train, y_train, epochs=100, validation_data=(X_test, y_test), verbose=0)
print("Навчання завершено!")

Початок навчання...
Навчання завершено!


In [20]:
# Тестування
test_sentences = [
    "Це було дуже погано",
    "Мені дуже подобається",
    "Сьогодні звичайний день"
]

In [21]:
# Обробка тестових фраз
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_padded = pad_sequences(test_seq, maxlen=max_length, padding=padding_type)

# Прогноз
predictions = model.predict(test_padded)
classes = ['Негативний', 'Нейтральний', 'Позитивний']

print("\n--- РЕЗУЛЬТАТИ ТЕСТУВАННЯ ---")
for i, sentence in enumerate(test_sentences):
    res_idx = np.argmax(predictions[i])
    conf = predictions[i][res_idx] * 100
    print(f"Речення: '{sentence}' -> Тон: {classes[res_idx]} ({conf:.1f}% впевненості)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step

--- РЕЗУЛЬТАТИ ТЕСТУВАННЯ ---
Речення: 'Це було дуже погано' -> Тон: Негативний (100.0% впевненості)
Речення: 'Мені дуже подобається' -> Тон: Позитивний (99.7% впевненості)
Речення: 'Сьогодні звичайний день' -> Тон: Нейтральний (99.9% впевненості)
